# IQP simulation and Torch gradients

Public QASP configuration, neural embedding, sampled expectations, exact TN execution and automatic backend selection. This tutorial generates its results locally; it does not need archived experiment data.

The [full comparison notebook](https://github.com/andre-sequeira10/QASP/blob/3bda8622bd6f5ddfb7c05a0dfd0538c1b0ccd718/examples/iqp_simulator.ipynb) and research results remain in the pinned upstream archive.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "qasp").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "qasp").is_dir()
sys.path.insert(0, str(ROOT))
import numpy as np
import torch
import pennylane as qml
import pandas as pd
import json
from dataclasses import asdict
from qasp import IQPSimulator
from qasp.rl.model import create_model
from qasp.genlearn import create_qcbm
torch.set_num_threads(1)
torch.manual_seed(2026)
print("Torch", torch.__version__, "PennyLane", qml.__version__)

from IPython.display import Image, display

def show_table(table, precision=2, missing="unavailable", formatters=None):
    """Format missing observations explicitly without altering numerical data."""
    display(table.style.format(formatter=formatters, precision=precision, na_rep=missing))


## The commuting construction

\[
U_X(\theta)=e^{-i\sum_j\theta_jX^{g_j}}
=H^{\otimes n}e^{-i\sum_j\theta_jZ^{g_j}}H^{\otimes n}.
\]
With physical rotation angles \(\phi_j=2\theta_j\), a Z correlation is
\[
\langle Z^a\rangle=
\mathbb E_{z\text{ uniform}}\cos\!\left[
\sum_j\phi_j(-1)^{g_j\cdot z}{\bf1}\{g_j\cdot a=1\pmod2\}\right].
\]
Encoding gates use the same family, with angles
\(\alpha_l(s)=\langle s,\lambda_l\rangle+b_l\).
The supplied neural encoder has no bias; a constant input coordinate implements
one. These circuits can entangle qubits.

We first average **all** uniform bits on a small circuit, checking the identity
and its gradients independently of Monte Carlo error. Enumeration is exponential
and is only a validation option.


In [ ]:
def circuit(theta, features):
    qml.RX(theta[0] * features[0], 0)
    qml.IsingXX(theta[1], [0, 1])
    qml.PauliRot(theta[2] + features[1], "XXX", [0, 1, 2])
    return qml.expval(qml.Z(0) + .4 * qml.Z(1) @ qml.Z(2))

theta = torch.tensor([.4, -.3, .7], dtype=torch.float64, requires_grad=True)
features = torch.tensor([.2, -.5], dtype=torch.float64, requires_grad=True)
devices = {
    "enumerated estimator": IQPSimulator(3, backend="van_den_nest", vn_sampling="enumerate"),
    "exact TN": IQPSimulator(3, backend="tn"),
    "full state": qml.device("default.qubit", wires=3),
}
checked = {}
for name, device in devices.items():
    node = qml.QNode(circuit, device, interface="torch", diff_method="backprop")
    value = node(theta, features)
    grads = torch.autograd.grad(value, (theta, features))
    checked[name] = (value.detach(), *(g.detach() for g in grads))
for name in ("enumerated estimator", "exact TN"):
    for actual, expected in zip(checked[name], checked["full state"]):
        torch.testing.assert_close(actual, expected, atol=2e-12, rtol=2e-12)
pd.DataFrame([{"backend": name, "value": float(result[0]),
               "parameter_gradient": result[1].tolist(),
               "input_gradient": result[2].tolist()}
              for name, result in checked.items()])


## Ordinary policy configuration with a neural encoder

This is the public `PQCSoftmax` factory, with compatible commuting RX, RXX and
RXXX layers. Set `backend` to `"auto"` or `"tn"` without changing the policy.
The example forces sampled correlations to demonstrate training through them.
Softmax/log losses of noisy correlations generally have finite-sample bias.


In [ ]:
n = 6
config = {
    "ansatz": "AngleAnsatz",
    "ansatz_config": {
        "n_layers": 2,
        "rotations": ([f"X {w}" for w in range(n)]
                      + [f"XX {w} {w+1}" for w in range(n-1)]
                      + [f"XXX {w} {w+1} {w+2}" for w in range(n-2)]),
        "include_imprimitive": False,
        "include_hadamards_initial": False,
        "reuploading": True,
    },
    "feature_map": "NeuralEmbedding",
    "feature_map_config": {"features": 3, "rotations": "X"},
    "n_actions": 2, "observables": [qml.Z(0), qml.Z(n-1)],
    "fc_layer": None, "device": "IQPSimulator",
    "device_kwargs": {"backend": "van_den_nest", "vn_samples": 8192, "seed": 2026},
    "diff_method": "backprop", "shots": None,
}
policy = create_model("PQCSoftmax", config, n_qubits=n).double()
optimizer = torch.optim.Adam(policy.parameters(), lr=.01)
history = []
for step in range(3):
    # The constant third coordinate gives each neural projection a bias weight.
    observation = torch.tensor([.3, -.4, 1.], dtype=torch.float64, requires_grad=True)
    optimizer.zero_grad()
    logits = policy(observation)
    loss = -torch.log_softmax(logits, dim=-1)[0]
    loss.backward()
    encoder = dict(policy.named_parameters())["qlayer.input_scaling_params"]
    assert encoder.grad is not None and torch.isfinite(encoder.grad).all()
    assert observation.grad is not None and torch.isfinite(observation.grad).all()
    history.append({"step": step, "loss": loss.item(),
                    "encoder_gradient_norm": encoder.grad.norm().item(),
                    "input_gradient_norm": observation.grad.norm().item()})
    optimizer.step()
pd.DataFrame(history)


In [ ]:
report = policy.circuit.device.last_report
{k: getattr(report, k) for k in (
    "selected", "approximate", "samples", "sampling",
    "standard_errors", "hoeffding_half_widths", "reason"
)}


## Accuracy budgets and exact-only execution

For \(H_r=\sum_kc_{rk}Z^{a_k}\), a simultaneous fixed-point bound across \(M\)
outputs is \(L_r\sqrt{2\log(2M/\delta)/S}\), where \(L_r=\sum_k|c_{rk}|\).
Shared bitstrings correlate term estimates; the empirical standard error of each
returned weighted sum accounts for this covariance. Bounds are on values,
not nonlinear losses or relative/gradient error.


In [ ]:
tape = qml.tape.make_qscript(circuit)(theta, features)
accurate = IQPSimulator(3, backend="van_den_nest", vn_error=.02, vn_confidence=.99)
plan = accurate.plan(tape, requires_grad=True)
exact_only = IQPSimulator(3, allow_approximate=False)
print("Required classical samples:", plan.samples)
print("Simultaneous value bounds:", plan.hoeffding_half_widths)
print("Exact-only selected:", exact_only.plan(tape).selected)


## Generative MMD uses independent estimates

A squared noisy moment is biased. QASP detects sampled classical estimates even
with `shots=None` and uses two independent executions for the model-square
term. This also overrides fixed-bank reuse inside that MMD evaluation.


In [ ]:
model = create_qcbm({
    "ansatz": "IQPAnsatz",
    "ansatz_config": {"n_layers": 1, "imprimitive_pattern": "chain"},
    "loss": "mmd", "mmd_config": {"mode": "exact"},
    "device": "IQPSimulator",
    "device_kwargs": {"backend": "van_den_nest", "vn_samples": 4096, "seed": 41},
    "diff_method": "backprop", "seed": 41,
}, n_qubits=3, data=[[0, 0, 0], [1, 1, 1]])
mmd_optimizer = torch.optim.Adam([model.params], lr=.01)
mmd_optimizer.zero_grad()
mmd_loss = model.loss()
mmd_loss.backward()
assert torch.isfinite(model.params.grad).all()
mmd_optimizer.step()
print("MMD estimate:", mmd_loss.item())
print("Estimator sampling:", model.device.last_report.sampling)


## How auto selects a backend

`IQPSimulator(backend="auto")` chooses between **QASP TN and QASP Van den Nest**, not the PennyLane devices. It checks eligibility and resource limits, estimates sampled work from the number of bits, gates, supports, moments and samples, and compares this with TN contraction work plus a small-operation overhead proxy. The report stores both scores and the reason.

Defaults are 4096 samples, a TN arithmetic threshold of 1e8, 256 MiB estimated memory and a 20000-operation overhead proxy per gate/wire and output network, adjusted for batching and gradients. For an eligible, viable estimator, more than 64 wires or more than 512 recorded gates, including basis changes, skips TN planning. This can bypass an inexpensive exact chain contraction; it is a heuristic planning limit, not runtime optimality. `allow_approximate=False` disables the sampled candidate and shortcut. Increasing the sample count can favor TN. Probability readouts exclude this estimator. The small statevector fallback remains available when neither IQP route qualifies, subject to its size limit.

The example below executes a 100-qubit chain and differentiates its moments. It demonstrates the API and selection report, not a performance benchmark.


In [ ]:
# Direct public-device example: 100-qubit commuting chain, complete Torch gradient.
n = 100
large_device = IQPSimulator(wires=n, backend="auto", vn_samples=4096, seed=2026)
@qml.qnode(large_device, interface="torch", diff_method="backprop")
def large_chain(angles):
    for j in range(n):
        qml.RX(angles[j], wires=j)
    for j in range(n - 1):
        qml.IsingXX(angles[n + j], wires=[j, j + 1])
    for j in range(n - 2):
        qml.PauliRot(angles[2*n - 1 + j], "XXX", wires=[j, j + 1, j + 2])
    return qml.expval(qml.Z(0)), qml.expval(qml.prod(*(qml.Z(j) for j in range(n))))
angles = torch.tensor(np.random.default_rng(2026).normal(0, .35, 3*n - 3), dtype=torch.float64, requires_grad=True)
outputs = torch.stack(large_chain(angles))
gradient = torch.autograd.grad(outputs @ torch.tensor([.2, 1.], dtype=torch.float64), angles)[0]
assert gradient.shape == angles.shape and torch.isfinite(gradient).all()
assert large_device.last_report.selected == "van_den_nest"
assert large_device.last_report.tn_planning_skipped
print("Moments:", outputs.detach().tolist(), "gradient entries:", gradient.numel())
print("Selection:", large_device.last_report.reason)


## Scope across QASP

- RL softmax and expectation classifiers can use the estimator for compatible Z moments.
- Projected quantum kernels can use Z-moment features; nonlinear kernel values need not be unbiased.
- Moment MMD uses independent replicas. IQP optimization objectives can use diagonal Hamiltonians.
- Born probabilities, general variances, and quantum output samples stay on TN or another eligible device.
- `auto` uses a bounded cost heuristic, not a runtime-optimality guarantee.
- Existing `IQPTNSimulator` and the legacy `device="auto"` retain their behavior.

See [the guide](../docs/iqp_simulator.md) for mathematical bounds, all sampling
options, and resource controls. The full test suite also checks a 1,000-wire
high-order rotation without constructing a statevector or TN.
